# 04 - Module 1: Fidelity

Applies the three fidelity metrics (Comprehensiveness, Sufficiency,
Infidelity) to every (model, XAI, dataset) configuration. Produces
`experiments/results/module1_fidelity_results.csv`.


In [ ]:
import numpy as np
import pandas as pd

from xai_blockchain_framework.config import CONFIG, PATHS, set_seed
from xai_blockchain_framework.metrics.fidelity import evaluate_fidelity
from xai_blockchain_framework.models import load_ml_model
from xai_blockchain_framework.utils import save_csv

set_seed()
K_VALUES = [1, 3, 5, 10]

ell = np.load(PATHS.data_processed / 'elliptic_splits.npz')
eth = np.load(PATHS.data_processed / 'ethereum_splits.npz')
ell_idx = np.load(PATHS.results_dir / 'xai_eval_indices_elliptic.npy')
eth_idx = np.load(PATHS.results_dir / 'xai_eval_indices_ethereum.npy')


## Loop over ML configurations

In [ ]:
rows = []
for xai in ['shap', 'lime']:
    for model_name in ['randomforest', 'lightgbm']:
        for ds, splits, indices in [
            ('Elliptic', ell, ell_idx),
            ('Ethereum', eth, eth_idx),
        ]:
            attrs = np.load(PATHS.results_dir / f'{xai}_{model_name}_{ds.lower()}.npy')
            model = load_ml_model(PATHS.models_dir / f'{ds.lower()}_{model_name}.joblib')
            predict = lambda X: model.predict_proba(X)[:, 1]  # noqa: E731
            df = evaluate_fidelity(
                predict, splits['X_test'], attrs, indices,
                k_values=K_VALUES,
                rng=np.random.default_rng(CONFIG.random_seed),
            )
            df['Type'] = 'ML'
            df['Model'] = model_name.upper()
            df['XAI'] = xai.upper()
            df['Dataset'] = ds
            rows.append(df)

results = pd.concat(rows, ignore_index=True)
cols = ['Type', 'Model', 'XAI', 'Dataset', 'k', 'Comprehensiveness', 'Sufficiency', 'Infidelity']
results = results[cols]
print(results.to_string(index=False))
save_csv(results, PATHS.results_dir / 'module1_fidelity_results.csv')
